# Test Predict — Telco Customer Churn
**Course:** Machine Learning  
**Authors:** Aye Khin Khin Hpone (Yolanda Lim) — ST125970 | Subhana Chitrakar — ST126138

---
Verifies `model.pkl` across 6 test scenarios:
1. High-risk churn customer
2. Low-risk customer (expected: No Churn)
3. Senior citizen, no internet
4. Edge case — tenure = 0
5. Batch prediction (4 customers)
6. Feature engineering consistency check

In [1]:
import pickle
import pandas as pd
import numpy as np

# Column order must match train.ipynb exactly
NUMERICAL_COLS   = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_month']
CATEGORICAL_COLS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
ALL_COLS = NUMERICAL_COLS + CATEGORICAL_COLS

pipeline = pickle.load(open('model.pkl', 'rb'))
print('model.pkl loaded.')
print(f'Steps: {list(pipeline.named_steps.keys())}')

model.pkl loaded.
Steps: ['preprocessor', 'classifier']


In [2]:
def predict_customer(data: dict) -> dict:
    """Mirrors app.py /predict logic exactly — including feature engineering."""
    tenure  = float(data.get('tenure', 0))
    monthly = float(data.get('MonthlyCharges', 0))
    total   = float(data.get('TotalCharges', 0))
    cpm     = total / (tenure + 1)  # feature engineering

    row = {'tenure': tenure, 'MonthlyCharges': monthly,
           'TotalCharges': total, 'charges_per_month': cpm}
    for col in CATEGORICAL_COLS:
        row[col] = str(data.get(col, ''))

    df   = pd.DataFrame([row], columns=ALL_COLS)
    pred = pipeline.predict(df)[0]
    prob = pipeline.predict_proba(df)[0]

    return {
        'prediction':    int(pred),
        'label':         'Churn' if pred == 1 else 'No Churn',
        'confidence':    round(float(max(prob)) * 100, 2),
        'prob_churn':    round(float(prob[1]) * 100, 2),
        'prob_no_churn': round(float(prob[0]) * 100, 2),
    }

print('Helper ready.')

Helper ready.


## Test 1 — High-Risk Churn Customer
Short tenure, month-to-month, fiber optic, no security, electronic check.

In [3]:
r = predict_customer({
    'tenure': 1, 'MonthlyCharges': 85.0, 'TotalCharges': 85.0,
    'gender': 'Male', 'SeniorCitizen': '0', 'Partner': 'No', 'Dependents': 'No',
    'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No',
    'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check'
})
print(f"Label: {r['label']}  |  P(Churn): {r['prob_churn']}%  |  Confidence: {r['confidence']}%")
assert r['prediction'] == 1, 'Expected Churn'
print('PASS')

Label: Churn  |  P(Churn): 70.38%  |  Confidence: 70.38%
PASS


## Test 2 — Low-Risk Customer
Long tenure, two-year contract, DSL, all security services, auto payment.

In [4]:
r = predict_customer({
    'tenure': 60, 'MonthlyCharges': 55.0, 'TotalCharges': 3300.0,
    'gender': 'Female', 'SeniorCitizen': '0', 'Partner': 'Yes', 'Dependents': 'Yes',
    'PhoneService': 'Yes', 'MultipleLines': 'Yes', 'InternetService': 'DSL',
    'OnlineSecurity': 'Yes', 'OnlineBackup': 'Yes', 'DeviceProtection': 'Yes',
    'TechSupport': 'Yes', 'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Two year', 'PaperlessBilling': 'No',
    'PaymentMethod': 'Bank transfer (automatic)'
})
print(f"Label: {r['label']}  |  P(Churn): {r['prob_churn']}%  |  Confidence: {r['confidence']}%")
assert r['prediction'] == 0, 'Expected No Churn'
print('PASS')

Label: No Churn  |  P(Churn): 1.42%  |  Confidence: 98.58%
PASS


## Test 3 — Senior Citizen, No Internet

In [5]:
r = predict_customer({
    'tenure': 24, 'MonthlyCharges': 25.0, 'TotalCharges': 600.0,
    'gender': 'Female', 'SeniorCitizen': '1', 'Partner': 'No', 'Dependents': 'No',
    'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'No',
    'OnlineSecurity': 'No internet service', 'OnlineBackup': 'No internet service',
    'DeviceProtection': 'No internet service', 'TechSupport': 'No internet service',
    'StreamingTV': 'No internet service', 'StreamingMovies': 'No internet service',
    'Contract': 'One year', 'PaperlessBilling': 'No', 'PaymentMethod': 'Mailed check'
})
print(f"Label: {r['label']}  |  P(Churn): {r['prob_churn']}%")
print('PASS — no error raised')

Label: No Churn  |  P(Churn): 3.29%
PASS — no error raised


## Test 4 — Edge Case: tenure = 0

In [6]:
r = predict_customer({
    'tenure': 0, 'MonthlyCharges': 70.0, 'TotalCharges': 0.0,
    'gender': 'Male', 'SeniorCitizen': '0', 'Partner': 'No', 'Dependents': 'No',
    'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No',
    'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check'
})
print(f"Label: {r['label']}  |  P(Churn): {r['prob_churn']}%")
print('PASS — tenure=0 handled correctly (charges_per_month = 0.0 / 1 = 0.0)')

Label: Churn  |  P(Churn): 94.25%
PASS — tenure=0 handled correctly (charges_per_month = 0.0 / 1 = 0.0)


## Test 5 — Batch Prediction (4 customers)

In [7]:
batch_input = [
    {'tenure':1,  'MonthlyCharges':85.0,  'TotalCharges':85.0,   'gender':'Male',   'SeniorCitizen':'0','Partner':'No', 'Dependents':'No', 'PhoneService':'Yes','MultipleLines':'No',             'InternetService':'Fiber optic','OnlineSecurity':'No',                  'OnlineBackup':'No',                  'DeviceProtection':'No',                 'TechSupport':'No',                 'StreamingTV':'No',                 'StreamingMovies':'No',                 'Contract':'Month-to-month','PaperlessBilling':'Yes','PaymentMethod':'Electronic check'},
    {'tenure':72, 'MonthlyCharges':45.0,  'TotalCharges':3240.0, 'gender':'Female', 'SeniorCitizen':'0','Partner':'Yes','Dependents':'Yes','PhoneService':'Yes','MultipleLines':'Yes',            'InternetService':'DSL',        'OnlineSecurity':'Yes',                 'OnlineBackup':'Yes',                 'DeviceProtection':'Yes',                'TechSupport':'Yes',                'StreamingTV':'No',                 'StreamingMovies':'No',                 'Contract':'Two year',       'PaperlessBilling':'No', 'PaymentMethod':'Credit card (automatic)'},
    {'tenure':12, 'MonthlyCharges':65.0,  'TotalCharges':780.0,  'gender':'Male',   'SeniorCitizen':'1','Partner':'No', 'Dependents':'No', 'PhoneService':'Yes','MultipleLines':'No',             'InternetService':'Fiber optic','OnlineSecurity':'No',                  'OnlineBackup':'No',                  'DeviceProtection':'No',                 'TechSupport':'No',                 'StreamingTV':'Yes',                'StreamingMovies':'Yes',                'Contract':'Month-to-month','PaperlessBilling':'Yes','PaymentMethod':'Electronic check'},
    {'tenure':36, 'MonthlyCharges':30.0,  'TotalCharges':1080.0, 'gender':'Female', 'SeniorCitizen':'0','Partner':'Yes','Dependents':'No', 'PhoneService':'No', 'MultipleLines':'No phone service','InternetService':'No',         'OnlineSecurity':'No internet service','OnlineBackup':'No internet service','DeviceProtection':'No internet service','TechSupport':'No internet service','StreamingTV':'No internet service','StreamingMovies':'No internet service','Contract':'One year',       'PaperlessBilling':'Yes','PaymentMethod':'Mailed check'},
]

rows = []
for d in batch_input:
    t = float(d['tenure']); tc = float(d['TotalCharges'])
    row = {'tenure': t, 'MonthlyCharges': float(d['MonthlyCharges']),
           'TotalCharges': tc, 'charges_per_month': tc / (t + 1)}
    for col in CATEGORICAL_COLS:
        row[col] = str(d[col])
    rows.append(row)

batch_df = pd.DataFrame(rows, columns=ALL_COLS)
preds    = pipeline.predict(batch_df)
probs    = pipeline.predict_proba(batch_df)

print(f'{"#":<4}{"Label":<12}{"P(Churn)":<12}{"Confidence"}')
print('-' * 40)
for i, (pred, prob) in enumerate(zip(preds, probs)):
    label = 'Churn' if pred == 1 else 'No Churn'
    print(f'{i+1:<4}{label:<12}{prob[1]*100:.2f}%{"":<6}{max(prob)*100:.2f}%')

assert len(preds) == 4
print('\nPASS — batch prediction complete')

#   Label       P(Churn)    Confidence
----------------------------------------
1   Churn       70.38%      70.38%
2   No Churn    1.53%      98.47%
3   Churn       90.17%      90.17%
4   No Churn    1.36%      98.64%

PASS — batch prediction complete


## Test 6 — Feature Engineering Consistency
Verifies charges_per_month formula is identical in train.ipynb, app.py, and this notebook.

In [ ]:
test_cases = [
    {'tenure': 0,  'TotalCharges': 0.0,    'expected': 0.0},
    {'tenure': 1,  'TotalCharges': 85.0,   'expected': 85.0 / 2},
    {'tenure': 12, 'TotalCharges': 780.0,  'expected': 780.0 / 13},
    {'tenure': 72, 'TotalCharges': 3240.0, 'expected': 3240.0 / 73},
]
for tc in test_cases:
    got = tc['TotalCharges'] / (tc['tenure'] + 1)
    assert abs(got - tc['expected']) < 1e-9
    print(f"tenure={tc['tenure']:2d}  TotalCharges={tc['TotalCharges']:7.1f}  "
          f"charges_per_month={got:.4f}  OK")
print('\nPASS — feature engineering consistent with train.ipynb and app.py')

tenure= 0  TotalCharges=    0.0  charges_per_month=0.0000  OK
tenure= 1  TotalCharges=   85.0  charges_per_month=42.5000  OK
tenure=12  TotalCharges=  780.0  charges_per_month=60.0000  OK
tenure=72  TotalCharges= 3240.0  charges_per_month=44.3836  OK

PASS — feature engineering consistent with train.ipynb and app.py


: 